In [ ]:
%matplotlib inline

# 04.3 Fractionation

## Contents

**Tools:** CADET-Process (Python).

**Python used:** Object configuration, component names and objective functions.

**Introduces:** `Fractionator`, performance metrics and `FractionationOptimizer`.

1. Performance indicators
2. Choose collection windows and inspect the pools
3. Optimize collection windows
4. Choose an alternative objective

Characterization establishes the model parameters. We now use the simulated outlet to decide what to collect and how to assess the resulting products.

```{note}
A chromatogram shows concentration over time. Integrating concentration times flow rate over each collection window gives the collected amount:

$$
m_{i} = \sum_{k=1}^{n_{\text{chrom}}} \sum_{j=1}^{n_{\text{frac}, k}^{i}}\int_{t_{\text{start}, j}}^{t_{\text{end}, j}} Q_k(t) \cdot c_{i,k}(t) dt, \\
$$

where $n_{\text{frac}, k}^{i}$ is the number of fractions considered for component $i$ in chromatogram $k$, and $n_{\text{chrom}}$ is the total number of chromatograms evaluated.
```

## Key performance indicators (KPI)

### Productivity
How much product is collected per volume of stationary phase and per time: a measure of how well the column is used.

$$
PR_{i} = \frac{m_i}{V_{\text{solid}} \cdot \Delta t_{\text{cycle}}},\\
$$
with $V_{\text{solid}}$: volume of stationary phase.

### Recovery
Which share of the injected amount of a component ends up in its product pool.

$$
Y_{i} = \frac{m_i}{m_{\text{feed}, i}},\\
$$
with $m_{\text{feed},i}$: injected amount of component $i$.

### Eluent consumption
How much solvent is needed per amount of collected product.

$$
EC_{i} = \frac{V_{\text{solvent}}}{m_i}.\\
$$
with $V_{\text{solvent}}$: solvent used during a cycle.

### Purity
Which share of a product pool is the target component; the rest are other components collected with it.

$$
PU_{i} = \frac{m_{i}^{i}}{\sum_{l=1}^{n_{\text{comp}}} m_{l}^{i}},\\
$$
where $n_{\text{comp}}$ is the number of mixture components and $m_{l}^{i}$ is the amount of component $l$ in product pool $i$.

## Choose collection windows with `Fractionator`

In **CADET-Process**, the `fractionation` module provides methods to calculate these performance indicators.

`Fractionator` assigns collection windows to a product pool for each component. It can combine several windows and product-outlet chromatograms in one pool.

First choose the collection windows manually, using the batch-elution process from the [examples collection](https://cadet-process.readthedocs.io/en/latest/examples/batch_elution/process.html).

In [ ]:
# Load the two-component batch-elution model used throughout this lesson.
from examples.batch_elution.process import process

The supplied model marks its feed inlet, eluent inlet and product outlet when adding them to the flow sheet. These roles determine the injected amount, solvent consumption and chromatogram used for fractionation:

```
flow_sheet.add_unit(feed, feed_inlet=True)
flow_sheet.add_unit(eluent, eluent_inlet=True)
flow_sheet.add_unit(outlet, product_outlet=True)
```

The product outlet supplies the **chromatogram**. `Fractionator` then assigns slices of that signal to product pools or waste.

In [ ]:
from CADETProcess.simulator import Cadet
process_simulator = Cadet()
simulation_results = process_simulator.simulate(process)

The outlet chromatogram shows where each collection window could start and end:

In [ ]:
_ = simulation_results.solution.outlet.outlet.plot()

`Fractionator` takes the product chromatograms and `process_meta`. The metadata supplies the feed amount, cycle time and column volume needed for the performance indicators.

In [ ]:
# Fractionator for the product chromatogram
from CADETProcess.fractionation import Fractionator
fractionator = Fractionator(
    simulation_results.chromatograms,
    process_meta=simulation_results.process.process_meta,
)

To add a fractionation event, the following arguments need to be provided:

- `event_name`: Name of the event.
- `target`: Component name, such as `"A"`, identifying the product pool. `-1` sends the stream to waste.
- `time`: Time of the event in **seconds**.
- `chromatogram`: Product-outlet name, such as `"outlet"`. Optional when the results contain just one product chromatogram.

As a first choice, we collect **A from 5 to 5.75 min** and **B from 6.5 to 9 min**. These are initial choices; the calculated pool purity will tell us whether they are suitable.

In [ ]:
# Collection windows for A and B
fractionator.add_fractionation_event('start_A', 'A', 5*60, 'outlet')
fractionator.add_fractionation_event('end_A', -1, 5.75*60)
fractionator.add_fractionation_event('start_B', 'B', 6.5*60)
fractionator.add_fractionation_event('end_B', -1, 9*60)

`fractionator.performance` contains the calculated product-pool metrics:

In [ ]:
# Performance of the pools
print(fractionator.performance)

Purity and recovery differ between the two pools. Narrow collection windows can improve purity while discarding some target component.

`plot_fraction_signal()` overlays the collection windows on the chromatogram:

In [ ]:
# Collection windows on the chromatogram
_ = fractionator.plot_fraction_signal()

## Optimize collection windows
`FractionationOptimizer` adjusts collection windows to maximize the collected amount while meeting the specified purity requirements. Each component can have its own requirement.

In [ ]:
# Fractionation optimizer
from CADETProcess.fractionation import FractionationOptimizer
fractionation_optimizer = FractionationOptimizer()
# Limit the allowed purity-constraint violation to 0.0001 (0.01 percentage points).
fractionation_optimizer.optimizer.catol = 1e-4

```{note}
`catol` is the numerical constraint tolerance. A result such as **0.948** must not be reported as meeting **0.95** at this tolerance. The achieved purity is reported in `.performance`; a strict product specification also needs a small safety margin. The default initial search radius (`rhobeg`) is sufficient here.
```

`optimize_fractionation()` takes the simulation results and purity requirements. First, we require **95% purity for A**; a zero requirement means B is not collected as a product.

In [ ]:
# Optimize for 95% purity of A
# purity_required has one entry per component, [A, B]; 0 means B is not collected
fractionator = fractionation_optimizer.optimize_fractionation(simulation_results, purity_required=[0.95, 0])

`optimize_fractionation()` returns a `Fractionator`; its `.performance` contains the achieved purity, recovery and other indicators.

In [ ]:
# Performance of the optimized pools
print(fractionator.performance)

The optimized collection windows differ from the manual choice:

In [ ]:
# Optimized collection windows
_ = fractionator.plot_fraction_signal()

With **95% purity required for B** and no product pool for A, the windows change:

In [ ]:
# Optimize for 95% purity of B
# purity_required: [A, B]
fractionator = fractionation_optimizer.optimize_fractionation(simulation_results, purity_required=[0, 0.95])

print(fractionator.performance)
_ = fractionator.plot_fraction_signal()

With **80% purity required for both A and B**, the optimizer keeps their collection windows separate, so the outlet stream is assigned to only one pool at a time.

In [ ]:
# Optimize for 80% purity of A and B
# purity_required: [A, B]
fractionator = fractionation_optimizer.optimize_fractionation(simulation_results, purity_required=[0.8, 0.8])

print(fractionator.performance)
_ = fractionator.plot_fraction_signal()

## Alternative objectives

A custom objective takes a `Performance` object and returns a score. Here, maximizing collected amount times concentration also discourages dilution of the product pool.

```{note}
The default optimizer, `COBYLA`, uses one objective. `RankedPerformance` combines the component metrics using weights $w_i$.
```

$$
p = \frac{\sum_i^{n_{comp}}w_i \cdot p_i}{\sum_i^{n_{comp}}(w_i)}
$$

The custom objective returns mass times concentration. `minimize=False` maximizes it.

In [ ]:
# Alternative objective: mass times concentration
from CADETProcess.performance import RankedPerformance
ranking = [1, 1]

def alternative_objective(performance):
    performance = RankedPerformance(performance, ranking)
    return performance.mass * performance.concentration

fractionator = fractionation_optimizer.optimize_fractionation(
    simulation_results, purity_required=[0.95, 0.95],
    obj_fun=alternative_objective, minimize=False,
)

In [ ]:
# Performance with the alternative objective
print(fractionator.performance)
_ = fractionator.plot_fraction_signal()

The resulting fractionation times show that in this case, it is advantageous to discard some slices of the peak in order not to dilute the overall product fraction.

## Summary

- `Fractionator` assigns collection windows to product pools and calculates their performance.
- `FractionationOptimizer` improves those windows while meeting purity requirements.
- A collection strategy is judged by productivity, purity and recovery together.

**Now:** [**04.3 Fractionation: Exercises**](04.3b_fractionation_exercises.ipynb): choose and optimize a collection window for one protein.

**Next lesson:** [**04.4 Process Optimization**](../04.4%20Process%20Optimization/04.4a_process_optimization_lesson.ipynb) also varies feed duration and cycle time.